# Train Driver Drowsiness Detection trên Google Colab (Stage 2 Custom ResNet-18 + Stage 3 Causal Temporal)

Notebook này triển khai đầy đủ quy trình huấn luyện trong `ARCHITECTURE.md` trên **Google Colab GPU**:
1. **Nạp repo & dữ liệu DDD**: tự động gắn Google Drive, giải nén dataset về SSD `/content/` để đọc nhanh, và ánh xạ đường dẫn trong `data/frames_manifest.csv` sang Linux path.
2. **Pha 1 — Cache Eye Crops theo `video_id` (`YOLOv8EyeCropper`)**: dùng `models/yolo_best.pt` cắt mắt trái/phải (`64x64`) + `eye_mask` (`0/1/2` mắt) một lần duy nhất và sao lưu cache lên Drive (nếu bật).
3. **Pha 2 — Huấn luyện Joint Stage 2 + Stage 3**: train `SharedEyeEncoder` (custom `ResNet18`) + `TemporalHead` (`gru`, `lstm`, hoặc `transformer`) theo chiến lược **Many-to-One** trên cửa sổ trượt nội bộ từng `video_id`, chia tập độc lập theo `subject_id`.
4. **Đánh giá độc lập trên tập `test`**: báo cáo `Accuracy`, `Precision`, `Recall`, `F1-Score`, `AUROC` ở cả mức **Window-level** và **Video-level**, kèm ma trận nhầm lẫn.

In [ ]:
# === 1. THIẾT LẬP MÔI TRƯỜNG COLAB & REPO ===
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

USE_GDRIVE = True  # Đặt False nếu bạn upload trực tiếp dữ liệu lên /content/
REPO_DIR = "/content/drive/MyDrive/AI_Project/driver_downsiness_detection"  # Đường dẫn thư mục code repo trên Colab/Drive
REPO_GIT_URL = ""  # Tùy chọn: điền link GitHub repo nếu muốn tự git clone khi REPO_DIR chưa tồn tại

if USE_GDRIVE:
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except ImportError:
        print("Không chạy trên Colab runtime, bỏ qua drive.mount().")

def resolve_repo_root() -> Path:
    candidates = [
        Path.cwd().resolve(),
        *Path.cwd().resolve().parents,
        Path(REPO_DIR),
        Path(REPO_DIR) / "driver_downsiness_detection",
        Path("/content/drive/MyDrive/driver_downsiness_detection"),
        Path("/content/drive/MyDrive/driver_downsiness_detection/driver_downsiness_detection"),
    ]
    for cand in candidates:
        if (cand / "configs" / "train.yaml").exists() and (cand / "models" / "yolo_best.pt").exists():
            return cand.resolve()

    if REPO_GIT_URL:
        subprocess.check_call(["git", "clone", REPO_GIT_URL, REPO_DIR])
        cloned = Path(REPO_DIR)
        if (cloned / "configs" / "train.yaml").exists():
            return cloned.resolve()
        nested = cloned / "driver_downsiness_detection"
        if (nested / "configs" / "train.yaml").exists():
            return nested.resolve()

    raise FileNotFoundError(
        "Không tìm thấy thư mục dự án (cần có configs/train.yaml và models/yolo_best.pt). "
        "Hãy upload repo hoặc điền REPO_DIR / REPO_GIT_URL."
    )

ROOT = resolve_repo_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

for pkg, pip_name in [("ultralytics", "ultralytics"), ("sklearn", "scikit-learn"), ("yaml", "pyyaml"), ("tqdm", "tqdm")]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

import torch
print(f"Repo root: {ROOT}")
print(f"PyTorch  : {torch.__version__} | CUDA={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

In [ ]:
# === 2. CẤU HÌNH TRUNG TÂM ===
CFG = {
    # Đường dẫn tới thư mục chứa {Drowsy/, Non Drowsy/} hoặc file .zip trên Drive/Colab
    "dataset_dir": None,  # vd: "/content/drive/MyDrive/Driver Drowsiness Dataset (DDD)"
    "dataset_zip": "/content/drive/MyDrive/AI_Project/DDD.zip",  # vd: "/content/drive/MyDrive/DDD.zip" (sẽ tự giải nén ra /content/ddd_raw)
    # Thư mục trên Google Drive để lưu cache eye_crops & checkpoints (để None nếu không muốn backup Drive)
    "drive_backup_dir": "/content/drive/MyDrive/drowsiness_checkpoints" if USE_GDRIVE else None,

    # Backbone Stage 2 muốn train: chọn 1 hoặc nhiều trong ["resnet18", "mobilenet_v3_small", "mobilenet_v3_large", "custom_resnet"]
    "backbones": ["resnet18", "mobilenet_v3_small"],
    "pretrained": True,
    "freeze_early_layers": True,

    # Kiến trúc Stage 3 muốn train: chọn 1 hoặc nhiều trong ["gru", "lstm", "transformer"]
    "archs": ["gru", "lstm", "transformer"],

    # Siêu tham số dữ liệu & cửa sổ trượt
    "seed": 42,
    "val_ratio": 0.2,
    "window_size": 30,
    "temporal_stride": 3,
    "window_step_train": 25,
    "window_step_eval": 15,

    # Siêu tham số huấn luyện GPU
    "extract_batch_size": 64 if torch.cuda.is_available() else 32,
    "epochs": 30,
    "batch_size": 32 if torch.cuda.is_available() else 16,
    "num_workers": 2 if torch.cuda.is_available() else 0,
    "backbone_lr": 1.0e-4,
    "temporal_lr": 3.0e-4,
    "weight_decay": 2.0e-2,
    "grad_clip_norm": 1.0,
    "freeze_backbone": False,
    "use_pos_weight": True,
    "augment": True,
    "label_smoothing": 0.05,
    "warmup_epochs": 3,
    "early_stopping_patience": 8,
    "preload_cache": True,
    "gpu_augment": True,
    "channels_last": True,
    "use_compile": False,  # Tắt mặc định để tránh tốn ~140s compile lần đầu trên T4
    "prefetch_factor": 2,
    "amp": torch.cuda.is_available(),
    "eval_threshold": 0.5,
    "selection_metric": "auroc",
    "cache_dir": "/content/eye_crops_cache" if Path("/content").exists() else str(ROOT / "data/cache/eye_crops"),
    "checkpoint_dir": "/content/checkpoints" if Path("/content").exists() else str(ROOT / "checkpoints"),
}
print("Config loaded:", {k: CFG[k] for k in ("backbones", "pretrained", "freeze_early_layers", "archs", "epochs", "batch_size")})

In [ ]:
# === 3. ÁNH XẠ ĐƯỜNG DẪN ẢNH TRÊN COLAB & KHÔI PHỤC CACHE (NẾU CÓ) ===
import csv
import shutil
import zipfile
from pathlib import PurePosixPath, PureWindowsPath

cache_dir = Path(CFG["cache_dir"])
cache_dir.mkdir(parents=True, exist_ok=True)

backup_dir = Path(CFG["drive_backup_dir"]) if CFG["drive_backup_dir"] else None
cache_zip_backup = (backup_dir / "eye_crops_cache.zip") if backup_dir else None
if cache_zip_backup is not None and cache_zip_backup.exists() and not any(cache_dir.glob("*.pt")):
    print(f"Đang giải nén eye_crops_cache từ Drive: {cache_zip_backup} ...")
    with zipfile.ZipFile(cache_zip_backup, "r") as zf:
        zf.extractall(cache_dir)
    print(f"Đã khôi phục {len(list(cache_dir.glob('*.pt')))} file cache .pt")

def find_ddd_dataset_dir() -> Path | None:
    if CFG["dataset_zip"] is not None:
        zip_path = Path(CFG["dataset_zip"]).expanduser()
        if not zip_path.exists():
            raise FileNotFoundError(f"Không thấy dataset_zip: {zip_path}")
        unzip_target = Path("/content/ddd_raw")
        if not (unzip_target / "Drowsy").exists():
            print(f"Đang giải nén {zip_path} -> {unzip_target} ...")
            unzip_target.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(unzip_target)
        for cand in [unzip_target, *unzip_target.iterdir()]:
            if cand.is_dir() and (cand / "Drowsy").exists() and (cand / "Non Drowsy").exists():
                return cand.resolve()

    candidates = []
    if CFG["dataset_dir"] is not None:
        candidates.append(Path(CFG["dataset_dir"]).expanduser())
    candidates.extend([
        ROOT / "data" / "raw",
        ROOT.parent / "Driver Drowsiness Dataset (DDD)",
        Path("/content/ddd_raw"),
        Path("/content/Driver Drowsiness Dataset (DDD)"),
        Path("/content/drive/MyDrive/Driver Drowsiness Dataset (DDD)"),
        Path("/content/drive/MyDrive/DDD"),
    ])
    for cand in candidates:
        if cand.exists() and (cand / "Drowsy").exists() and (cand / "Non Drowsy").exists():
            return cand.resolve()
    return None

DATASET_DIR = find_ddd_dataset_dir()
src_manifest = ROOT / "data" / "frames_manifest.csv"
colab_manifest = ROOT / "data" / "frames_manifest_colab.csv"

if DATASET_DIR is not None:
    print(f"Dataset DDD dir: {DATASET_DIR}")
    remapped_rows = []
    missing_count = 0
    with src_manifest.open("r", encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            raw_p = row["image_path"]
            p_obj = PureWindowsPath(raw_p) if ":" in raw_p or "\\" in raw_p else PurePosixPath(raw_p)
            new_p = DATASET_DIR / p_obj.parent.name / p_obj.name
            if not new_p.exists():
                missing_count += 1
            row["image_path"] = new_p.as_posix()
            remapped_rows.append(row)

    with colab_manifest.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["video_id", "frame_idx", "label", "image_path"])
        writer.writeheader()
        writer.writerows(remapped_rows)

    print(f"Đã tạo {colab_manifest.name}: {len(remapped_rows)} frames | missing={missing_count}")
    if missing_count > 0:
        raise FileNotFoundError(f"Thiếu {missing_count} ảnh trong {DATASET_DIR}!")
    MANIFEST_PATH = colab_manifest
else:
    cached_pts = list(cache_dir.glob("*.pt"))
    if len(cached_pts) >= 54:
        print(f"Không thấy ảnh raw nhưng đã đủ {len(cached_pts)} file cache .pt -> dùng manifest gốc.")
        MANIFEST_PATH = src_manifest
    else:
        raise FileNotFoundError(
            "Không tìm thấy dataset ảnh DDD (Drowsy/ & Non Drowsy/) và chưa có đủ cache .pt. "
            "Hãy điền CFG['dataset_dir'] hoặc CFG['dataset_zip']."
        )

In [ ]:
# === 4. PHA 1: TRÍCH XUẤT & CACHE EYE CROPS BẰNG YOLOv8 ===
import matplotlib.pyplot as plt
from configs.vision import load_vision_config
from data.dataset import load_video_records
from tools.extract_features import cache_eye_crops_for_videos

vision_cfg = load_vision_config(ROOT / "configs" / "vision.yaml")
records = load_video_records(ROOT / "data" / "labels.csv", MANIFEST_PATH)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

newly_cached = cache_eye_crops_for_videos(
    records=records,
    cache_dir=cache_dir,
    detector_config=vision_cfg.detector,
    batch_size=CFG["extract_batch_size"],
    overwrite=False,
    device=device,
)
print(f"Đã trích xuất mới: {newly_cached} videos | Tổng cache hiện có: {len(list(cache_dir.glob('*.pt')))} / {len(records)} videos")

# Backup cache lên Google Drive để các lần chạy sau không phải chạy lại YOLOv8
if backup_dir is not None and (newly_cached > 0 or (cache_zip_backup is not None and not cache_zip_backup.exists())):
    backup_dir.mkdir(parents=True, exist_ok=True)
    print(f"Đang sao lưu cache lên Drive: {cache_zip_backup} ...")
    with zipfile.ZipFile(cache_zip_backup, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for pt_file in sorted(cache_dir.glob("*.pt")):
            zf.write(pt_file, arcname=pt_file.name)
    print("Sao lưu cache hoàn tất.")

# Kiểm tra trực quan 4 mẫu eye crop đã cache
sample_pts = sorted(cache_dir.glob("*.pt"))[:4]
fig, axes = plt.subplots(len(sample_pts), 2, figsize=(5, 2.2 * len(sample_pts)))
for row_idx, pt_path in enumerate(sample_pts):
    payload = torch.load(pt_path, map_location="cpu", weights_only=True)
    mid = payload["eye_crops"].shape[0] // 2
    crops = payload["eye_crops"][mid]  # (2, 3, 64, 64) uint8
    masks = payload["eye_mask"][mid].tolist()
    for slot, side in enumerate(["Left", "Right"]):
        ax = axes[row_idx, slot] if len(sample_pts) > 1 else axes[slot]
        ax.imshow(crops[slot].permute(1, 2, 0).numpy())
        ax.set_title(f"{payload['video_id']} f#{mid+1} {side} (v={masks[slot]:.0f})", fontsize=9)
        ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# === 5. PHA 2: HUẤN LUYỆN STAGE 2 (PRETRAINED RESNET18 / MOBILENETV3) + STAGE 3 (TEMPORAL HEAD) ===
from dataclasses import replace
from configs.temporal import load_temporal_config
from configs.train import TrainConfig
from data.dataset import preload_eye_crop_cache
from engine.train_temporal import run_training

TEMPORAL_YAML_MAP = {
    "gru": ROOT / "configs" / "temporal_gru.yaml",
    "lstm": ROOT / "configs" / "temporal_lstm.yaml",
    "transformer": ROOT / "configs" / "temporal_transformer.yaml",
}

# Preload cache 1 lần vào RAM và dùng chung cho toàn bộ tổ hợp (backbone x temporal)
shared_ram_cache = None
if CFG["preload_cache"]:
    train_val_ids = [r.video_id for r in records if r.split == "train"]
    shared_ram_cache = preload_eye_crop_cache(train_val_ids, cache_dir)
    print(f"Đã nạp sẵn {len(shared_ram_cache)} video vào RAM dùng chung cho {CFG['backbones']} x {CFG['archs']}.")

trained_checkpoints = {}
for bb_arch in CFG["backbones"]:
    bb_key = bb_arch.lower()
    run_vision_cfg = replace(
        vision_cfg,
        backbone=replace(
            vision_cfg.backbone,
            arch=bb_key,
            pretrained=CFG["pretrained"],
            freeze_early_layers=CFG["freeze_early_layers"],
        ),
    )
    for arch in CFG["archs"]:
        arch_key = arch.lower()
        run_tag = f"{bb_key}_{arch_key}"
        temporal_yaml = TEMPORAL_YAML_MAP[arch_key]
        temporal_cfg = load_temporal_config(temporal_yaml)

        train_cfg = TrainConfig(
            labels_csv=str(ROOT / "data" / "labels.csv"),
            frames_manifest_csv=str(MANIFEST_PATH),
            cache_dir=str(cache_dir),
            val_ratio=CFG["val_ratio"],
            window_size=CFG["window_size"],
            temporal_stride=CFG["temporal_stride"],
            window_step_train=CFG["window_step_train"],
            window_step_eval=CFG["window_step_eval"],
            vision_config=str(ROOT / "configs" / "vision.yaml"),
            temporal_config=str(temporal_yaml),
            seed=CFG["seed"],
            input_mode="crop_cache",
            auto_build_cache=False,
            extract_batch_size=CFG["extract_batch_size"],
            epochs=CFG["epochs"],
            batch_size=CFG["batch_size"],
            num_workers=CFG["num_workers"],
            backbone_lr=CFG["backbone_lr"],
            temporal_lr=CFG["temporal_lr"],
            weight_decay=CFG["weight_decay"],
            grad_clip_norm=CFG["grad_clip_norm"],
            freeze_backbone=CFG["freeze_backbone"],
            use_pos_weight=CFG["use_pos_weight"],
            augment=CFG["augment"],
            label_smoothing=CFG["label_smoothing"],
            warmup_epochs=CFG["warmup_epochs"],
            early_stopping_patience=CFG["early_stopping_patience"],
            preload_cache=CFG["preload_cache"],
            gpu_augment=CFG["gpu_augment"],
            channels_last=CFG["channels_last"],
            use_compile=CFG["use_compile"],
            prefetch_factor=CFG["prefetch_factor"],
            amp=CFG["amp"],
            eval_threshold=CFG["eval_threshold"],
            selection_metric=CFG["selection_metric"],
            checkpoint_dir=CFG["checkpoint_dir"],
            device="cuda" if torch.cuda.is_available() else "cpu",
        )

        print(f"\n==================== TRAINING: {bb_key.upper()} + {arch_key.upper()} ====================")
        best_ckpt = run_training(
            train_cfg=train_cfg,
            vision_cfg=run_vision_cfg,
            temporal_cfg=temporal_cfg,
            preloaded_cache=shared_ram_cache,
        )
        trained_checkpoints[run_tag] = best_ckpt

        if backup_dir is not None:
            ckpt_backup_dir = backup_dir / "checkpoints"
            ckpt_backup_dir.mkdir(parents=True, exist_ok=True)
            copied = []
            for suffix in ("best", "last"):
                src_ckpt = Path(CFG["checkpoint_dir"]) / f"{run_tag}_{suffix}.pt"
                if src_ckpt.exists():
                    dst_ckpt = ckpt_backup_dir / src_ckpt.name
                    shutil.copy2(src_ckpt, dst_ckpt)
                    copied.append(f"{dst_ckpt.name} ({dst_ckpt.stat().st_size / 1e6:.1f} MB)")
            if hasattr(os, "sync"):
                os.sync()
            print(f"Đã sao lưu {', '.join(copied)} -> {ckpt_backup_dir}")

In [ ]:
# === 6. ĐÁNH GIÁ ĐỘC LẬP TRÊN TẬP TEST (7 SUBJECTS / 14 VIDEOS) ===
import math
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from engine.evaluate import evaluate_checkpoint

summary_rows = []
n_runs = len(trained_checkpoints)
ncols = min(3, max(1, n_runs))
nrows = math.ceil(n_runs / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4.0 * nrows))
axes_flat = [axes] if n_runs == 1 else list(axes.flat)

for ax, (run_tag, ckpt_path) in zip(axes_flat, trained_checkpoints.items()):
    pred_csv = Path(CFG["checkpoint_dir"]) / f"{run_tag}_test_preds.csv"
    res = evaluate_checkpoint(
        checkpoint_path=ckpt_path,
        split="test",
        threshold=CFG["eval_threshold"],
        save_csv=pred_csv,
    )
    if backup_dir is not None and pred_csv.exists():
        shutil.copy2(pred_csv, backup_dir / "checkpoints" / pred_csv.name)

    w = res["window_metrics"]
    v = res["video_metrics"]
    summary_rows.append({
        "backbone": res["backbone"],
        "arch": res["arch"],
        "win_acc": round(w["accuracy"], 4),
        "win_prec": round(w["precision"], 4),
        "win_rec": round(w["recall"], 4),
        "win_f1": round(w["f1"], 4),
        "win_auroc": round(w["auroc"], 4),
        "vid_acc": round(v["accuracy"], 4),
        "vid_f1": round(v["f1"], 4),
        "vid_auroc": round(v["auroc"], 4),
    })

    df_pred = pd.read_csv(pred_csv)
    cm = confusion_matrix(df_pred["label"], df_pred["pred"], labels=[0, 1])
    disp = ConfusionMatrixDisplay(cm, display_labels=["Non Drowsy", "Drowsy"])
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"{run_tag.upper()} Test")

for unused_ax in axes_flat[n_runs:]:
    unused_ax.axis("off")

plt.tight_layout()
plt.show()

df_summary = pd.DataFrame(summary_rows).sort_values(by=["vid_auroc", "win_auroc"], ascending=False)
display(df_summary)
best_tag = f"{df_summary.iloc[0]['backbone']}_{df_summary.iloc[0]['arch']}"
print(f"Chi tiết dự đoán từng video của cấu hình tốt nhất ({best_tag}):")
display(pd.read_csv(Path(CFG["checkpoint_dir"]) / f"{best_tag}_test_preds.csv"))

In [ ]:
# === 7. ĐỒNG BỘ GOOGLE DRIVE & TỰ ĐỘNG NGẮT COLAB RUNTIME ===
import time

AUTO_DISCONNECT = True  # Đặt False nếu muốn giữ runtime mở sau khi chạy xong

if hasattr(os, "sync"):
    os.sync()

if AUTO_DISCONNECT:
    try:
        from google.colab import drive, runtime  # type: ignore

        if USE_GDRIVE:
            print("Đang flush dữ liệu lên Google Drive...")
            drive.flush_and_unmount()
        print("Hoàn tất train & eval. Tự động ngắt Colab runtime sau 5 giây...")
        time.sleep(5)
        runtime.unassign()
    except ImportError:
        print("Không chạy trên Google Colab, bỏ qua tự động ngắt runtime.")